# EX_00 — PyTorch vs TensorFlow (ejercicios)

**Notebook de referencia:** `notebook/00_Frameworks_Pytorch_vs_Tensorflow.ipynb`

**Tiempo orientativo:** ~30 minutos.

En esta hoja practicarás ideas equivalentes en ambos frameworks: tensores, capas lineales y un forward pass mínimo.


## Actividad 1 — Activación a mano

Implementa en NumPy una función `relu` y otra `sigmoid` y comprueba que coinciden con `torch` y `tensorflow` en un vector de prueba.

*Hint:* use `torch.relu`, `tf.nn.relu`; for sigmoid use `torch.sigmoid` and `tf.nn.sigmoid`.


In [1]:
import numpy as np
import torch
import tensorflow as tf

# TODO: implement relu_np(z) and sigmoid_np(z)
def relu_np(z):
    # np.maximum compara elemento a elemento con el 0
    return np.maximum(0, z)

def sigmoid_np(z):
    # np.exp calcula e^(-z) de forma vectorizada
    return 1 / (1 + np.exp(-z))

# Vector de prueba dado por el ejercicio
x = np.array([-2.0, 0.0, 1.5], dtype=np.float32)

# --- Comprobación de ReLU ---
out_relu_np = relu_np(x)
out_relu_torch = torch.relu(torch.tensor(x)).numpy()
out_relu_tf = tf.nn.relu(x).numpy()

# --- Comprobación de Sigmoid ---
out_sigmoid_np = sigmoid_np(x)
out_sigmoid_torch = torch.sigmoid(torch.tensor(x)).numpy()
out_sigmoid_tf = tf.nn.sigmoid(x).numpy()

# TODO: assert close to torch and tensorflow outputs
# np.testing.assert_allclose verifica que los arrays sean prácticamente idénticos (tolerando decimales mínimos flotantes)
np.testing.assert_allclose(out_relu_np, out_relu_torch, rtol=1e-5)
np.testing.assert_allclose(out_relu_np, out_relu_tf, rtol=1e-5)

np.testing.assert_allclose(out_sigmoid_np, out_sigmoid_torch, rtol=1e-5)
np.testing.assert_allclose(out_sigmoid_np, out_sigmoid_tf, rtol=1e-5)

print("¡Todo coincide a la perfección!")
print(f"Resultados ReLU: {out_relu_np}")
print(f"Resultados Sigmoid: {out_sigmoid_np}")


¡Todo coincide a la perfección!
Resultados ReLU: [0.  0.  1.5]
Resultados Sigmoid: [0.11920293 0.5        0.8175745 ]


## Actividad 2 — Misma arquitectura, dos APIs

Define una red `Linear(10, 3)` + `ReLU` + `Linear(3, 1)` en **PyTorch** (`nn.Sequential`) y la misma en **Keras** (`Sequential`).

*Hint:* set seeds (`torch.manual_seed`, `tf.random.set_seed`) and use explicit init if you want to compare weights.


In [2]:
import torch
import torch.nn as nn
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Fijamos las semillas por si quieres que los pesos aleatorios sean reproducibles
torch.manual_seed(42)
tf.random.set_seed(42)

# =====================================================================
# 1. TODO: build torch_model and keras_model with the architecture above
# =====================================================================

# Modelo en PyTorch usando nn.Sequential
torch_model = nn.Sequential(
    nn.Linear(10, 3),  # En PyTorch se especifica (input_features, output_features)
    nn.ReLU(),
    nn.Linear(3, 1)
)

# Modelo en Keras usando keras.Sequential
keras_model = keras.Sequential([
    # En Keras el primer argumento son las unidades de salida. 
    # Especificamos input_shape=[10] para que la capa sepa que recibe 10 características.
    layers.Dense(3, activation='relu', input_shape=[10]), 
    layers.Dense(1)
])


# =====================================================================
# 2. TODO: run a forward pass on random input shape (batch=4, features=10)
# =====================================================================

# Creamos la entrada aleatoria con tamaño (batch=4, features=10)
# Para PyTorch (necesita un tensor de tipo float)
x_torch = torch.randn(4, 10)

# Para Keras (necesita un tensor de TensorFlow o un array de NumPy)
x_tf = tf.random.normal([4, 10])

# Ejecutamos el Forward Pass (pasar los datos a través de la red)
output_torch = torch_model(x_torch)
output_keras = keras_model(x_tf)

# Imprimimos los resultados para verificar las dimensiones de salida (deberían ser [4, 1])
print("Dimensión de salida en PyTorch:", output_torch.shape)
print("Dimensión de salida en Keras:", output_keras.shape)

print("\nValores de salida PyTorch:\n", output_torch)
print("\nValores de salida Keras:\n", output_keras)


Dimensión de salida en PyTorch: torch.Size([4, 1])
Dimensión de salida en Keras: (4, 1)

Valores de salida PyTorch:
 tensor([[0.3805],
        [0.0316],
        [0.4628],
        [0.1774]], grad_fn=<AddmmBackward0>)

Valores de salida Keras:
 tf.Tensor(
[[ 0.3037356 ]
 [-0.26271823]
 [ 0.2808807 ]
 [ 0.29493886]], shape=(4, 1), dtype=float32)


c:\Users\carlo\anaconda3\envs\IA_MUBA\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


## Actividad 3 — Entrenamiento en mini-batch (conceptual + código corto)

Escribe un bucle de **una época** que: (1) muestree un batch sintético `X, y` para regresión, (2) calcule `MSE`, (3) haga `backward` / `gradient` y un paso de optimizador.

Elige **solo uno** de los dos frameworks para el bucle completo; en el otro, documenta en un comentario qué API usarías (`loss.backward`, `tape.gradient`, etc.).


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim

# --- CONFIGURACIÓN INICIAL ---
# Creamos un modelo lineal simple y un optimizador en PyTorch
model = nn.Linear(10, 1)
criterion = nn.MSELoss()  # Error Cuadrático Medio para regresión
optimizer = optim.SGD(model.parameters(), lr=0.01)

# Simulamos los datos de una época repartidos en 5 mini-batches
# Cada batch tendrá un tamaño de (batch_size=4, features=10)
num_batches = 5
batch_size = 4

print("Iniciando la época de entrenamiento...")

# =====================================================================
# TODO: one epoch, one framework (PyTorch)
# =====================================================================
for batch_idx in range(num_batches):
    # 1. Muestrear un batch sintético X, y para regresión
    X_batch = torch.randn(batch_size, 10)
    y_batch = torch.randn(batch_size, 1)
    
    # Resetear los gradientes acumulados del paso anterior
    optimizer.zero_grad()
    
    # Forward pass: calcular la predicción
    predictions = model(X_batch)
    
    # 2. Calcular el MSE (Loss)
    loss = criterion(predictions, y_batch)
    
    # 3. Hacer backward (cálculo de gradientes)
    loss.backward()
    
    # Paso del optimizador (actualizar los pesos de la red)
    optimizer.step()
    
    print(f"  -> Batch {batch_idx + 1}/{num_batches} - Loss (MSE): {loss.item():.4f}")


# =====================================================================
# TODO: comment the other API (TensorFlow / Keras)
# =====================================================================
"""
DOCUMENTACIÓN DE LA API DE TENSORFLOW (Equivalente conceptual):

Si hubiéramos elegido TensorFlow/Keras para el bucle, la API clave a usar es `tf.GradientTape()`.
El flujo dentro de cada batch habría sido el siguiente:

1. Muestrear los datos como tensores de TF: 
   X_batch = tf.random.normal([batch_size, 10])
   y_batch = tf.random.normal([batch_size, 1])

2. Calcular la pérdida dentro del contexto del grabador de gradientes ('tape'):
   with tf.GradientTape() as tape:
       predictions = tf_model(X_batch)
       loss = tf.keras.losses.mean_squared_error(y_batch, predictions) # MSE

3. Calcular los gradientes con respecto a los pesos entrenables del modelo:
   gradients = tape.gradient(loss, tf_model.trainable_variables)

4. Aplicar los gradientes usando el optimizador de Keras:
   tf_optimizer.apply_gradients(zip(gradients, tf_model.trainable_variables))
"""

print("\n¡Época completada con éxito!")


Iniciando la época de entrenamiento...
  -> Batch 1/5 - Loss (MSE): 0.3302
  -> Batch 2/5 - Loss (MSE): 0.6067
  -> Batch 3/5 - Loss (MSE): 1.3494
  -> Batch 4/5 - Loss (MSE): 0.4774
  -> Batch 5/5 - Loss (MSE): 0.4912

¡Época completada con éxito!
